In [1]:
from _bootstrap import setup
from src.vasijas.reporting import ProcessingReport

import cv2
import matplotlib.pyplot as plt
import numpy as np
from pathlib import Path
from enum import StrEnum
import os
import pandas as pd

In [2]:
class OrientacionCorte(StrEnum):
    VERTICAL = "vertical"
    HORIZONTAL = "horizontal"

In [3]:
def obtener_o_crear_directorio(ruta_base: Path, carpeta_destino: str):
    ruta_completa = ruta_base / carpeta_destino

    ruta_completa.mkdir(parents=True, exist_ok=True)

    if not ruta_completa.exists():
        raise RuntimeError("No se pudo crear el directorio")

    return ruta_completa


# Funciones
def obtener_nombre_carpeta_clase(valor_clase: str | int | float):
    """
    Devuelve el nombre de la carpeta según la 'Clase Ana':
    - Si hay número → 'clase01', 'clase02', ... 'clase11', etc.
    - Si está vacío/NaN → 'sin clase'
    """
    if pd.isna(valor_clase) or str(valor_clase).strip() == "":
        return "sin_clase"
    try:
        numero = int(float(valor_clase))  # maneja '11.0' como 11
        return f"clase{numero:02d}"
    except ValueError:
        return "sin_clase"



In [4]:
def obtener_rutas_perfiles(
        ruta_origen: Path, clase: int | str, extension: str
) -> list[Path]:
    nombre_carpeta = obtener_nombre_carpeta_clase(clase)
    carpeta = ruta_origen / nombre_carpeta

    return list(carpeta.glob(f"*.{extension}"))

In [5]:
def determinar_orientacion_corte(clase: str) -> OrientacionCorte:
    if clase in range(1, 10):
        return OrientacionCorte.VERTICAL
    return OrientacionCorte.HORIZONTAL


def crear_ruta_archivo(ruta_base: Path, nombre_archivo: str, sufijo: str):
    return os.path.join(ruta_base, f"{nombre_archivo}{sufijo}.png")

In [6]:
def mostrar_imagen(imagen):
    plt.imshow(imagen, cmap=plt.cm.gray)
    plt.axis("on")
    plt.figure()


def obtener_nombre_archivo(ruta: Path) -> str:
    return Path(ruta).stem


def guardar_imagen(ruta: Path, imagen):
    cv2.imwrite(ruta, imagen)


def leer_imagen(ruta_imagen: str):
    img = cv2.imread(ruta_imagen, cv2.IMREAD_GRAYSCALE)
    return img


def cortar_imagen(img, orientacion_corte: OrientacionCorte):
    alto, ancho = img.shape

    if orientacion_corte == OrientacionCorte.VERTICAL:
        tope_slice = (slice(None, alto // 2), slice(None))
        base_slice = (slice(alto // 2, None), slice(None))
    else:
        tope_slice = (slice(None), slice(None, ancho // 2))
        base_slice = (slice(None), slice(ancho // 2, None))

    fondo_tope = np.full_like(img, 255)
    fondo_base = np.full_like(img, 255)

    fondo_tope[tope_slice] = img[tope_slice]
    fondo_base[base_slice] = img[base_slice]

    return (fondo_tope, fondo_base)



In [7]:
def procesar_perfil(ruta_perfil: Path, orientacion: OrientacionCorte, ruta_destino_clase: Path) -> None:
    """Procesa un perfil individual: lo carga, corta y guarda las mitades."""
    img = leer_imagen(ruta_perfil)
    tope, base = cortar_imagen(img, orientacion)
    nombre_original = obtener_nombre_archivo(ruta_perfil)
    guardar_imagen(ruta=crear_ruta_archivo(ruta_destino_clase, nombre_original, "_tope"), imagen=tope, )
    guardar_imagen(ruta=crear_ruta_archivo(ruta_destino_clase, nombre_original, "_base"), imagen=base, )


def obtener_identificador_desde_ruta(ruta_perfil: Path) -> str:
    # Ajustá si tu lógica de ID es otra
    return ruta_perfil.stem


def procesar_clase(
    ruta_origen: Path,
    ruta_destino: Path,
    clase: int | str,
    meta_indexado: pd.DataFrame,
    columna_id: str,
    registros_nuevos: list[dict],
    report: ProcessingReport | None = None,
) -> None:
    """Procesa todos los perfiles de una clase específica y genera metadata de mitades."""
    perfiles = obtener_rutas_perfiles(ruta_origen, clase, extension="png")
    if not perfiles:
        return

    if report is not None:
        report.registrar_perfiles_en_clase(clase, len(perfiles))

    orientacion = determinar_orientacion_corte(clase)
    ruta_destino_clase = obtener_o_crear_directorio(
        ruta_destino,
        obtener_nombre_carpeta_clase(clase),
    )

    for ruta_perfil in perfiles:
        identificador = obtener_identificador_desde_ruta(ruta_perfil)

        # Buscamos la fila original en el metadata
        try:
            base_meta_row = meta_indexado.loc[identificador]
        except KeyError:
            # Si no encontrás metadata para ese perfil, lo podés reportar como error
            if report is not None:
                report.registrar_error(
                    clase=clase,
                    ruta_perfil=ruta_perfil,
                    exc=KeyError(f"Identificador {identificador!r} no encontrado en metadata"),
                )
            continue

        procesar_perfil(
            ruta_perfil=ruta_perfil,
            orientacion=orientacion,
            ruta_destino_clase=ruta_destino_clase,
            clase=clase,
            base_meta_row=base_meta_row,
            columna_id=columna_id,
            registros_nuevos=registros_nuevos,
            report=report,
        )


def procesar_imagenes(
    ruta_origen: Path,
    ruta_destino: Path,
    clases: list[int | str],
    metadata_original: pd.DataFrame,
    columna_id: str = "Identificador",
    report: ProcessingReport | None = None,
) -> tuple[ProcessingReport, pd.DataFrame]:
    """
    Orquesta el procesamiento de imágenes para múltiples clases.

    Devuelve:
      - report: ProcessingReport con el resumen del procesamiento
      - df_mitades: DataFrame con las filas de metadata para las imágenes cortadas
    """
    if report is None:
        report = ProcessingReport()

    # indexamos el metadata por la columna de identificador
    meta_indexado = metadata_original.set_index(columna_id)

    registros_nuevos: list[dict] = []

    for clase in clases:
        procesar_clase(
            ruta_origen=ruta_origen,
            ruta_destino=ruta_destino,
            clase=clase,
            meta_indexado=meta_indexado,
            columna_id=columna_id,
            registros_nuevos=registros_nuevos,
            report=report,
        )

    df_mitades = pd.DataFrame(registros_nuevos)
    return report, df_mitades


In [8]:
from _bootstrap import setup
import pandas as pd

PATHS = setup()

ruta_metadata = PATHS.DATA_CLASIFICADOS / "metadata.csv"
metadata_original = pd.read_csv(ruta_metadata)

clases = [9]

report = ProcessingReport()
report, df_mitades = procesar_imagenes(
    ruta_origen=PATHS.DATA_CLASIFICADOS,
    ruta_destino=PATHS.DATA_MITADES,
    clases=clases,
    metadata_original=metadata_original,
    columna_id="Identificador",
    report=report,
)

print(report.resumen_texto())

metadata_ampliado = pd.concat([metadata_original, df_mitades], ignore_index=True)

ruta_metadata_mitades = PATHS.DATA / "metadata_mitades.csv"
metadata_ampliado.to_csv(ruta_metadata_mitades, index=False)

'Futuro reporte'